# GraphRAG Affaires: demo

Questions over a knowledge graph of recent French political-financial affairs.

Prerequisites: Neo4j running (`docker compose up -d neo4j`), graph built (`make ingest-cached && make topics`),
API running (`make run`).

In [1]:
import json
import requests

API = "http://localhost:8000"

def show(data, limit=2500):
    print(json.dumps(data, ensure_ascii=False, indent=2)[:limit])

show(requests.get(f"{API}/health").json())
show(requests.get(f"{API}/api/stats").json())

{
  "status": "ok",
  "neo4j": "up"
}
{
  "nodes": {
    "Entity": 1466,
    "Chunk": 656,
    "Person": 577,
    "Institution": 354,
    "Company": 178,
    "Offense": 154,
    "Affair": 80,
    "Media": 70,
    "Document": 56,
    "Party": 53,
    "Topic": 17
  },
  "relationships": {
    "MENTIONS": 4497,
    "IN_TOPIC": 1466,
    "HAS_CHUNK": 656,
    "NEXT": 600,
    "LINKS_TO": 509,
    "INVOLVED_IN": 331,
    "WORKED_FOR": 289,
    "INDICTED_IN": 156,
    "CONVICTED_IN": 112,
    "CONCERNS": 83,
    "MEMBER_OF": 76,
    "REVEALED": 59,
    "ACQUITTED_IN": 57,
    "ABOUT": 56,
    "INVESTIGATED": 42
  }
}


## 1. A question answered from the graph

The response contains the answer, the cited sources (passages `C..`, relations `R..`, paths `P..`),
the graph paths used and every tool call of the agent.

In [2]:
def ask(question):
    r = requests.get(f"{API}/api/ask", params={"q": question}, timeout=180).json()
    print("Q:", r["question"])
    print("Status:", r["status"], "| confidence:", r["confidence"], "| guardrail:", r["guardrail"])
    print("\nA:", r["answer"])
    print("\nGraph paths:")
    for p in r["graph_paths"]:
        print("  ", p)
    print("\nSources:")
    for s in r["sources"]:
        where = f"{s.get('document')} / {s.get('section')}" if s["kind"] == "passage" else s["kind"]
        print(f"  [{s['id']}] {where}: {s['text'][:160]}")
    print("\nTool calls:")
    for c in r["tool_calls"]:
        print(f"   {c['tool']:18} {json.dumps(c['args'], ensure_ascii=False)[:90]} -> {c['results']}")
    return r

_ = ask("Marine Le Pen a-t-elle été condamnée dans l'affaire des assistants parlementaires du FN ?")

Q: Marine Le Pen a-t-elle été condamnée dans l'affaire des assistants parlementaires du FN ?
Status: answered | confidence: 1.0 | guardrail: None

A: Oui, Marine Le Pen a été **condamnée définitivement** dans l'affaire des assistants parlementaires du FN. En **2025**, elle a été reconnue coupable de détournement de fonds publics en première instance (C1, R7, R9) et cette condamnation a été **confirmée en appel en juillet 2026**, bien que les peines aient été allégées (C1, R4, R12). Elle reste donc **condamnée en dernier ressort**, avec des peines de prison et d'inéligibilité partiellement exécutées.

Graph paths:
   Marine Le Pen -[CONVICTED_IN appeal, final, 2026-07]-> Affaire des assistants parlementaires du Front national au Parlement européen
   Marine Le Pen -[CONVICTED_IN first_instance, 2025-03]-> Affaire des assistants parlementaires du Front national au Parlement européen
   Marine Le Pen -[CONVICTED_IN first_instance, 2025-03]-> Détournement de fonds publics
   Marine Le Pen 

In [3]:
_ = ask("Quel lien entre Nicolas Sarkozy et Claude Guéant ?")

Q: Quel lien entre Nicolas Sarkozy et Claude Guéant ?
Status: answered | confidence: 0.98 | guardrail: None

A: Nicolas Sarkozy et Claude Guéant sont liés par l'**affaire Sarkozy-Kadhafi**, dans laquelle ils ont tous deux été mis en examen et condamnés pour des faits similaires. Selon les sources, Claude Guéant aurait reçu des **valises d'argent libyen** (via Alexandre Djouhri et Béchir Saleh) en 2005-2006, en lien avec un **financement présumé de la campagne présidentielle de Sarkozy en 2007** ([C1], [C2], [C3], [R17], [R20]). Les deux hommes étaient impliqués dans un **réseau de corruption et de trafic d’influence** autour de la Libye, avec des négociations secrètes incluant des marchés d’armement et des avantages financiers ([C3], [C4], [C6]). Guéant a été **condamné en 2025 pour corruption passive et trafic d’influence** ([R17]), tandis que Sarkozy a été **mis en examen pour les mêmes faits** ([C1], [R11]). Leur collaboration dans cette affaire est documentée par des échanges épist

In [4]:
_ = ask("Qui a révélé l'affaire Fillon et quand ?")

Q: Qui a révélé l'affaire Fillon et quand ?
Status: answered | confidence: 1.0 | guardrail: None

A: L'affaire Fillon a été révélée par *Le Canard enchaîné* le **24 janvier 2017** avec un article publié le 25 janvier, affirmant que Penelope Fillon était rémunérée pour un emploi présumé fictif ([R1]).

Graph paths:
   Le Canard enchaîné -[REVEALED 2017-01-24]-> Affaire Fillon

Sources:
  [R1] relation: Le Canard enchaîné -[REVEALED 2017-01-24]-> Affaire Fillon

Tool calls:
   plan               {"entities": ["affaire Fillon"], "tools": ["entity_relations"]} -> 1
   search_passages    {"query": "Qui a révélé l'affaire Fillon et quand ?"} -> 6
   find_entities      {"names": ["affaire Fillon"]} -> 2
   entity_relations   {"entity": "Affaire Fillon"} -> 12


## 2. Guardrail: questions outside the corpus

The first is refused by the similarity threshold without any LLM call; the second reaches the
answer step, where the LLM declares the sources insufficient.

In [5]:
_ = ask("Quelle est la recette de la tarte tatin ?")

Q: Quelle est la recette de la tarte tatin ?
Status: not_in_corpus | confidence: 0.0 | guardrail: no passage or relation relevant enough

A: Je ne trouve pas d'information à ce sujet dans le corpus (articles Wikipédia sur les affaires politico-financières françaises récentes).

Graph paths:

Sources:

Tool calls:
   plan               {"entities": [], "tools": []} -> 0
   search_passages    {"query": "Quelle est la recette de la tarte tatin ?"} -> 6
   find_entities      {"names": []} -> 0


In [6]:
_ = ask("Qui a gagné la coupe du monde de football 2018 ?")

Q: Qui a gagné la coupe du monde de football 2018 ?
Status: not_in_corpus | confidence: 0.0 | guardrail: the LLM judged the sources insufficient

A: Je ne trouve pas d'information à ce sujet dans le corpus (articles Wikipédia sur les affaires politico-financières françaises récentes).

Graph paths:

Sources:

Tool calls:
   plan               {"entities": ["coupe du monde de football 2018"], "tools": []} -> 1
   search_passages    {"query": "Qui a gagné la coupe du monde de football 2018 ?"} -> 6
   find_entities      {"names": ["coupe du monde de football 2018"]} -> 2
   entity_relations   {"entity": "Le Monde"} -> 8


## 3. Topics (Louvain) and related concepts

In [7]:
topics = requests.get(f"{API}/api/topics").json()
for t in topics[:10]:
    print(f"{t['id']:>5}  {t['size']:>3} entities  {t['name']:<45} {', '.join(t['top_entities'][:4])}")

  586  255 entities  Affaire Sarkozy-Kadhafi                       Nicolas Sarkozy, Affaire Sarkozy-Kadhafi, Affaire Bygmalion, Mediapart
  236  148 entities  Affaires Benalla                              Emmanuel Macron, Affaires Benalla, Alexandre Benalla, Le Monde
  567  144 entities  Affaire Carlos Ghosn                          Carlos Ghosn, Affaire Carlos Ghosn, Nissan, abus de biens sociaux
  239  130 entities  Affaire des assistants parlementaires du Front national au Parlement européen Affaire des assistants parlementaires du Front national au Parlement européen, Détournement de fonds publics, Marine Le Pen, Rassemblement national
  944  116 entities  Affaire Karachi                               Affaire Karachi, Ziad Takieddine, Édouard Balladur, Nicolas Bazire
  373  115 entities  Affaire Fillon                                François Fillon, Affaire Fillon, Le Canard enchaîné, Les Républicains
  931  107 entities  Affaire Balkany                               Parquet nation

In [8]:
graph = requests.get(f"{API}/api/topics/{topics[0]['id']}/graph", params={"depth": 2}).json()
print(len(graph["nodes"]), "nodes,", len(graph["edges"]), "edges")
names = {n["key"]: n["name"] for n in graph["nodes"]}
for e in graph["edges"][:15]:
    stage = f" ({e['stage']})" if e.get("stage") and e["stage"] != "unknown" else ""
    print(f"  {names[e['source']]} -[{e['type']}{stage}]-> {names[e['target']]}")

150 nodes, 98 edges
  Baghdadi al-Mahmoudi -[INVOLVED_IN]-> Affaire Sarkozy-Kadhafi
  Police judiciaire de Nice -[INVESTIGATED]-> Affaire Bygmalion
  Philippe Blanchetier -[CONVICTED_IN (first_instance)]-> Financement illégal de campagne électorale
  Philippe Blanchetier -[INVOLVED_IN]-> Affaire Bygmalion
  Le Point -[REVEALED]-> Affaire Bygmalion
  Guillaume Peltier -[WORKED_FOR]-> Bygmalion
  Guillaume Peltier -[WORKED_FOR]-> Com1+
  UMP -[INDICTED_IN (investigation)]-> Affaire Bygmalion
  UMP -[INVOLVED_IN]-> Affaire Bygmalion
  Bachir Saleh -[INVOLVED_IN]-> Affaire Sarkozy-Kadhafi
  Thierry Herzog -[CONVICTED_IN (first_instance)]-> Violation du secret de l'instruction
  Thierry Herzog -[CONVICTED_IN (first_instance)]-> trafic d’influence
  Thierry Herzog -[CONVICTED_IN (first_instance)]-> Affaire des écoutes
  Thierry Herzog -[CONVICTED_IN (appeal)]-> Affaire des écoutes
  Thierry Herzog -[INDICTED_IN (first_instance)]-> violation du secret professionnel


## 4. What the schema makes possible: direct Cypher

Legal outcomes are typed relations with a judicial stage, and offenses are nodes.

In [9]:
import os
from neo4j import GraphDatabase
from dotenv import load_dotenv

load_dotenv()
driver = GraphDatabase.driver(os.environ.get("NEO4J_URI", "bolt://localhost:7687"),
                              auth=(os.environ["NEO4J_USER"], os.environ["NEO4J_PASSWORD"]))

def cypher(query, **params):
    for record in driver.execute_query(query, **params).records:
        print(dict(record))

Convictions, with their stage and whether they are final:

In [10]:
cypher("""
MATCH (p:Person)-[r:CONVICTED_IN]->(a:Affair)
RETURN p.name AS person, a.name AS affair, r.stage AS stage, r.final AS final, r.date AS date, r.support AS support
ORDER BY r.support DESC LIMIT 15
""")

{'person': 'Marine Le Pen', 'affair': 'Affaire des assistants parlementaires du Front national au Parlement européen', 'stage': 'appeal', 'final': True, 'date': '2026-07', 'support': 4}
{'person': 'Patrick Balkany', 'affair': 'Affaire Balkany', 'stage': 'appeal', 'final': True, 'date': '2020-03-04', 'support': 4}
{'person': 'Isabelle Balkany', 'affair': 'Affaire Balkany', 'stage': 'appeal', 'final': True, 'date': '2020', 'support': 3}
{'person': 'Ziad Takieddine', 'affair': 'Affaire Karachi', 'stage': 'first_instance', 'final': False, 'date': '2020-06', 'support': 3}
{'person': 'Penelope Fillon', 'affair': 'Affaire Fillon', 'stage': 'appeal', 'final': True, 'date': '2022', 'support': 3}
{'person': 'Marine Le Pen', 'affair': 'Affaire des assistants parlementaires du Front national au Parlement européen', 'stage': 'first_instance', 'final': False, 'date': '2025-03', 'support': 2}
{'person': 'Fernand Le Rachinel', 'affair': 'Affaire des assistants parlementaires du Front national au Parle

Affairs that share an offense (one hop through `Offense` nodes):

In [11]:
cypher("""
MATCH (a:Affair)-[:CONCERNS]->(o:Offense)
WITH o, collect(DISTINCT a.name) AS affairs
WHERE size(affairs) > 1
RETURN o.name AS offense, affairs
ORDER BY size(affairs) DESC LIMIT 8
""")

{'offense': 'Détournement de fonds publics', 'affairs': ['Affaire Fillon', 'Affaire des assistants parlementaires du Front national au Parlement européen', 'Affaire des assistants parlementaires du Mouvement démocrate au Parlement européen', 'Affaire Karachi', "Affaire des sondages de l'Élysée", 'Affaire des primes en liquide']}
{'offense': 'favoritisme', 'affairs': ['Affaire Business France', 'Affaire McKinsey', "Affaire des sondages de l'Élysée", 'Affaire Bygmalion']}
{'offense': 'trafic d’influence', 'affairs': ['Affaire Fillon', 'Affaire Carlos Ghosn', 'Affaire des écoutes', 'Affaire Alstom']}
{'offense': 'Corruption', 'affairs': ['Affaire Carlos Ghosn', 'Affaire des écoutes', 'Affaire Karachi', 'Affaire Balkany']}
{'offense': 'abus de confiance', 'affairs': ['Affaire des assistants parlementaires du Front national au Parlement européen', 'Affaire des assistants parlementaires du Mouvement démocrate au Parlement européen', 'Fonds Marianne', 'Affaire Bygmalion']}
{'offense': 'compli

Every relation keeps its provenance: the passages and quotes it was extracted from.

In [12]:
cypher("""
MATCH (s:Entity)-[r:INDICTED_IN|CONVICTED_IN|ACQUITTED_IN]->(t:Entity)
RETURN s.name AS source, type(r) AS relation, t.name AS target, r.evidence[0] AS quote, r.chunk_ids AS chunks
ORDER BY r.confidence DESC LIMIT 5
""")

{'source': 'Catherine Griset', 'relation': 'INDICTED_IN', 'target': "recel d'abus de confiance", 'quote': "sa cheffe de cabinet, Catherine Griset, est mise en examen pour recel d'abus de confiance.", 'chunks': ['10687214-5']}
{'source': 'François Fillon', 'relation': 'INDICTED_IN', 'target': 'Affaire Fillon', 'quote': 'sa mise en examen', 'chunks': ['10615237-0', '10615237-3']}
{'source': 'Marc Joulaud', 'relation': 'INDICTED_IN', 'target': 'Affaire Fillon', 'quote': "Le 20 mars 2017, Marc Joulaud annonce qu'il va être convoqué en vue d'une mise en examen", 'chunks': ['10615237-19']}
{'source': 'Carlos Ghosn', 'relation': 'INDICTED_IN', 'target': 'Affaire Carlos Ghosn', 'quote': "Le mois suivant, il est inculpé pour utilisation de fonds de l'entreprise à des fins personnelles.", 'chunks': ['101131-2', '13052386-1', '13052386-2', '13052386-3', '13052386-24', '13052386-28', '13052386-56']}
{'source': 'Marine Le Pen', 'relation': 'INDICTED_IN', 'target': 'Affaire des assistants parlementa

## 5. Extraction quality

In [13]:
show(json.load(open("data/ingest_report.json")))

{
  "documents": 56,
  "chunks": 656,
  "entities": 1466,
  "relations_extracted": 4463,
  "relations_rejected": {
    "wrong_types": 1313,
    "unknown_entity": 518,
    "evidence_not_in_passage": 748
  },
  "relations_kept": 1770,
  "relation_edges": 1205
}
